## Code Interpreter: Выполнение Python-кода внутри агента

Как мы знаем, LLM не очень хорошо справляется с некоторыми задачами, такими, как вычисления. Для преодоления проблем с неточностью вычислений мы можем использовать Tool Calling, предоставив LLM инструмент для вычисления произвольных выражений, или даже выполнения произвольного кода на Python. Однако, чтобы такие вычисления были безопасными, нужно в идеале выполнять такой код в отдельной *песочнице*.

В этом ноутбуке мы разберём встроенный инструмент **Code Interpreter** в Yandex AI Studio на примере чат-бота для расчёта калорийности блюд. Но для начала попробуем реализовать это с помощью обычного Function Calling.

Для начала установим необходимые библиотеки:


In [ ]:
%pip install openai pandas openpyxl matplotlib python-dotenv


**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

И ещё полезная функция на будущее:


In [51]:
from IPython.display import Markdown, display

def printx(string):
    display(Markdown(string))


## Авторизация и создание клиента OpenAI

Для работы с языковыми моделями нам понадобится авторизоваться в Yandex Cloud. Для доступа к модели необходимы:

* идентификатор каталога `folder_id`
* API-ключ сервисного аккаунта `api_key`. Для работы с агентами и инструментами обычно достаточно ролей `ai.assistants.editor` и `ai.languageModels.user`.

Мы предполагаем, что соответствующие значения хранятся в переменных окружения. Удобно использовать два подхода:

* если вы разворачиваете код в Yandex Datasphere - используйте секреты проекта;
* если вы запускаете проект со своего компьютера - разместите значения ключей в файле `.env` в корне репозитория.

Обратите внимание: сессии Code Interpreter содержат код, данные и результаты вычислений, поэтому для таких сценариев особенно полезны модели с большим контекстным окном. В примерах ниже мы используем Qwen235B.


In [52]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

## Подготовим небольшую таблицу пищевой ценности

Чтобы все три примера были сопоставимы, создадим небольшой учебный каталог продуктов. В нём будут указаны:

* калории на 100 г;
* белки на 100 г;
* жиры на 100 г;
* углеводы на 100 г.

Значения ниже являются **приблизительными** и нужны только для демонстрации механики работы агента. Это не медицинская база знаний и не официальный справочник по питанию.

In [53]:
import pandas as pd

food_df = pd.DataFrame(
    [
        {"food": "Куриная грудка", "kcal": 165, "protein_g": 31.0, "fat_g": 3.6, "carbs_g": 0.0},
        {"food": "Яйцо куриное", "kcal": 143, "protein_g": 12.6, "fat_g": 9.5, "carbs_g": 0.7},
        {"food": "Гречка варёная", "kcal": 110, "protein_g": 4.2, "fat_g": 1.1, "carbs_g": 21.3},
        {"food": "Рис варёный", "kcal": 130, "protein_g": 2.7, "fat_g": 0.3, "carbs_g": 28.2},
        {"food": "Овсянка сухая", "kcal": 379, "protein_g": 13.2, "fat_g": 6.5, "carbs_g": 67.7},
        {"food": "Лосось", "kcal": 208, "protein_g": 20.0, "fat_g": 13.0, "carbs_g": 0.0},
        {"food": "Творог 5%", "kcal": 121, "protein_g": 17.0, "fat_g": 5.0, "carbs_g": 3.0},
        {"food": "Банан", "kcal": 89, "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 22.8},
        {"food": "Яблоко", "kcal": 52, "protein_g": 0.3, "fat_g": 0.2, "carbs_g": 14.0},
        {"food": "Картофель варёный", "kcal": 87, "protein_g": 1.9, "fat_g": 0.1, "carbs_g": 20.1},
        {"food": "Огурец", "kcal": 15, "protein_g": 0.8, "fat_g": 0.1, "carbs_g": 3.0},
        {"food": "Оливковое масло", "kcal": 884, "protein_g": 0.0, "fat_g": 100.0, "carbs_g": 0.0},
    ]
)

food_df.to_csv("food_catalog.csv", index=False, encoding="utf-8")

food_df


,food,kcal,protein_g,fat_g,carbs_g
0,Куриная грудка,165,31.0,3.6,0.0
1,Яйцо куриное,143,12.6,9.5,0.7
2,Гречка варёная,110,4.2,1.1,21.3
3,Рис варёный,130,2.7,0.3,28.2
4,Овсянка сухая,379,13.2,6.5,67.7
5,Лосось,208,20.0,13.0,0.0
6,Творог 5%,121,17.0,5.0,3.0
7,Банан,89,1.1,0.3,22.8
8,Яблоко,52,0.3,0.2,14.0
9,Картофель варёный,87,1.9,0.1,20.1


Для удобства превратим таблицу в Markdown:

In [54]:
def dataframe_to_markdown(dataframe: pd.DataFrame) -> str:
    columns = list(dataframe.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in dataframe.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, separator, *rows])


nutrition_markdown = dataframe_to_markdown(food_df)
printx("### Таблица продуктов\n" + nutrition_markdown)

### Таблица продуктов
| food | kcal | protein_g | fat_g | carbs_g |
| --- | --- | --- | --- | --- |
| Куриная грудка | 165 | 31.0 | 3.6 | 0.0 |
| Яйцо куриное | 143 | 12.6 | 9.5 | 0.7 |
| Гречка варёная | 110 | 4.2 | 1.1 | 21.3 |
| Рис варёный | 130 | 2.7 | 0.3 | 28.2 |
| Овсянка сухая | 379 | 13.2 | 6.5 | 67.7 |
| Лосось | 208 | 20.0 | 13.0 | 0.0 |
| Творог 5% | 121 | 17.0 | 5.0 | 3.0 |
| Банан | 89 | 1.1 | 0.3 | 22.8 |
| Яблоко | 52 | 0.3 | 0.2 | 14.0 |
| Картофель варёный | 87 | 1.9 | 0.1 | 20.1 |
| Огурец | 15 | 0.8 | 0.1 | 3.0 |
| Оливковое масло | 884 | 0.0 | 100.0 | 0.0 |

## Пример 1. Ручное выполнение Python через Function Calling

Начнём с самого «ручного» сценария. Здесь у модели **нет** собственного интерпретатора Python. Вместо этого мы описываем обычную функцию `run_python_expression`, и модель просит наше приложение выполнить её.

Это очень похоже на стандартный Function Calling:

* мы описываем инструмент через JSON Schema;
* модель решает, когда его вызвать;
* наш код получает аргументы;
* мы сами исполняем Python и возвращаем результат обратно модели.

В этом примере таблица продуктов будет встроена прямо в системный промпт. То есть модель получит данные в текстовом виде и на их основе сгенерирует Python-код.

❗ **Важно:** ниже используется `exec` в учебных целях. Такой подход удобен, чтобы показать механику Function Calling, но в production-сценариях выполнять произвольный код от модели без жёсткой песочницы и валидации нельзя.

Функция ниже ожидает, что в выражении результат будет присваиваться переменной `result` - для этого она описывается в `local_scope`.

In [55]:
def run_python_expression(code: str) -> dict:
    safe_builtins = {
        "sum": sum,
        "len": len,
        "round": round,
        "min": min,
        "max": max,
    }
    local_scope = {"result": None}

    try:
        exec(code, {"__builtins__": safe_builtins}, local_scope)
        return {
            "status": "success",
            "result": local_scope.get("result"),
            "executed_code": code,
        }
    except Exception as exc:
        return {
            "status": "error",
            "error": str(exc),
            "executed_code": code,
        }

Сначала опишем инструмент для Responses API. Затем сформируем системный промпт, в который встроим нашу таблицу продуктов. Обратите внимание на ключевую инструкцию: мы просим модель **не считать в уме**, а всегда генерировать Python-код и вызывать функцию `run_python_expression`.


In [56]:
run_python_tool = {
    "type": "function",
    "name": "run_python_expression",
    "description": "Выполняет учебный фрагмент Python-кода и возвращает значение переменной result.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "Python-код. Код обязан положить результат работы в переменную result."
            }
        },
        "required": ["code"]
    }
}


function_calling_instructions = f"""
Ты — ассистент-нутрициолог. Твоя задача — рассчитывать калорийность и БЖУ блюд.

Используй только следующую таблицу пищевой ценности на 100 граммов:

{nutrition_markdown}

Правила работы:
1. Для всех вычислений используй только инструмент run_python_expression.
2. Не считай значения в уме, даже если пример кажется простым.
3. Используй только продукты из таблицы.
4. Масштабируй значения пропорционально массе в граммах.
5. Генерируй только простой код на Python, без импортов и библиотек.
6. Итоговый Python-код должен положить в переменную result словарь следующего вида:
   {{
       "totals": {{
           "kcal": ...,
           "protein_g": ...,
           "fat_g": ...,
           "carbs_g": ...
       }},
       "comment": "..."
   }}
6. После получения результата от инструмента объясни ответ пользователю простым языком.
"""

user_says = (
    "Посчитай калории и БЖУ для блюда: 100 г яйца куриного, "
    "150 г гречки варёной, 120 г куриной грудки и 80 г огурца."
)

res = client.responses.create(
    model=model,
    instructions=function_calling_instructions,
    input=user_says,
    tools=[run_python_tool],
)

print("Получены следующие ответы:", [item.type for item in res.output])


Получены следующие ответы: ['function_call']


Теперь посмотрим, запросила ли модель вызов нашей функции. Если всё прошло правильно, в ответе должен появиться объект `function_call`, а внутри него — строка с Python-кодом.


In [57]:
function_call_item = None

for output_item in res.output:
    if output_item.type == "function_call":
        function_call_item = output_item
        print(f"Модель запросила вызов функции: {output_item.name}")
        print("\nАргументы функции:")
        print(output_item.arguments)

if function_call_item is None:
    print("Модель не вызвала функцию. В таком случае стоит усилить инструкцию или уточнить запрос.")


Модель запросила вызов функции: run_python_expression

Аргументы функции:
{"code": "kcal = (100/100)*143 + (150/100)*110 + (120/100)*165 + (80/100)*15\nprotein_g = (100/100)*12.6 + (150/100)*4.2 + (120/100)*31.0 + (80/100)*0.8\nfat_g = (100/100)*9.5 + (150/100)*1.1 + (120/100)*3.6 + (80/100)*0.1\ncarbs_g = (100/100)*0.7 + (150/100)*21.3 + (120/100)*0.0 + (80/100)*3.0\nresult = {\n    \"totals\": {\n        \"kcal\": round(kcal, 1),\n        \"protein_g\": round(protein_g, 1),\n        \"fat_g\": round(fat_g, 1),\n        \"carbs_g\": round(carbs_g, 1)\n    },\n    \"comment\": \"Расчёт выполнен на основе указанных продуктов и их масс. Все значения масштабированы на 100 грамм и просуммированы.\"\n}"}


Дальше происходит то, что в случае Function Calling всегда делает ваше приложение: оно самостоятельно исполняет функцию, получает результат и отправляет его обратно модели через `previous_response_id`.

Именно поэтому Function Calling даёт полный контроль, но и требует большего количества кода на стороне приложения.


In [58]:
import json

if function_call_item is not None:
    function_args = json.loads(function_call_item.arguments)
    tool_result = run_python_expression(**function_args)

    print("Результат выполнения Python-кода:")
    print(tool_result)

    res1 = client.responses.create(
        model=model,
        previous_response_id=res.id,
        input=(
            "Результат выполнения функции run_python_expression: "
            + json.dumps(tool_result, ensure_ascii=False)
        ),
    )

    printx(res1.output_text)

Результат выполнения Python-кода:
{'status': 'success', 'result': {'totals': {'kcal': 518.0, 'protein_g': 56.7, 'fat_g': 15.6, 'carbs_g': 35.1}, 'comment': 'Расчёт выполнен на основе указанных продуктов и их масс. Все значения масштабированы на 100 грамм и просуммированы.'}, 'executed_code': 'kcal = (100/100)*143 + (150/100)*110 + (120/100)*165 + (80/100)*15\nprotein_g = (100/100)*12.6 + (150/100)*4.2 + (120/100)*31.0 + (80/100)*0.8\nfat_g = (100/100)*9.5 + (150/100)*1.1 + (120/100)*3.6 + (80/100)*0.1\ncarbs_g = (100/100)*0.7 + (150/100)*21.3 + (120/100)*0.0 + (80/100)*3.0\nresult = {\n    "totals": {\n        "kcal": round(kcal, 1),\n        "protein_g": round(protein_g, 1),\n        "fat_g": round(fat_g, 1),\n        "carbs_g": round(carbs_g, 1)\n    },\n    "comment": "Расчёт выполнен на основе указанных продуктов и их масс. Все значения масштабированы на 100 грамм и просуммированы."\n}'}


Вот итоговый расчёт калорий и БЖУ для указанного блюда:

### 🍽️ Состав блюда:
- 100 г куриного яйца (варёного)  
- 150 г варёной гречки  
- 120 г куриной грудки (отварной/приготовленной без жира)  
- 80 г свежего огурца  

---

### 🔢 Результат:
| Показатель       | Значение         |
|------------------|------------------|
| **Калории**      | **518 ккал**     |
| **Белки**        | **56,7 г**       |
| **Жиры**         | **15,6 г**       |
| **Углеводы**     | **35,1 г**       |

---

### 💡 Примечание:
- Блюдо богато белком, умеренно по калориям и хорошо подойдёт для сбалансированного питания, поддержания мышечной массы или похудения.
- Углеводы в основном поступают из гречки, белки — из курицы и яйца, жиры — в основном от яйца.
- Огурец практически не вносит калорий, но добавляет объём и клетчатку.

Если нужно — могу предложить варианты уменьшения или увеличения калорийности.

## Function Calling vs Code Interpreter

Казалось бы, мы поборолись с проблемой вычислений. Но есть одна проблема.

**Function Calling** даёт приложению полный контроль. Модель решает, какую функцию вызвать и с какими аргументами, но сам код выполняете вы. Это удобно, когда у вас уже есть своя бизнес-логика, ограничения безопасности и доступ к внутренним сервисам. Но возникает необходимость сторого контроля за выполняемым кодом, поскольку в случае промпт-инъекции модель может сгенерировать потенциально зловредный код. Для этого в нашем примере мы ограничили список функций, которые можно использовать внутри `exec`.

**Code Interpreter** даёт модели собственную среду Python. Модель может:

* писать и запускать код;
* исправлять его при ошибках;
* работать с CSV и XLSX;
* строить графики и сохранять артефакты в файлы.

В этом и состоит главная разница: при Function Calling вы строите цикл исполнения сами, а при Code Interpreter платформа AI Studio берёт на себя исполнение Python-кода внутри изолированного контейнера.

## Пример 2. Built-in Code Interpreter с автоматическим контейнером

Теперь сделаем почти ту же задачу, но уже встроенным инструментом `code_interpreter`.

В этом режиме модель получает собственную среду Python в изолированном контейнере. Нам уже не нужно:

* вручную принимать `function_call`;
* отдельно вызывать `exec`;
* отдельно отправлять результат вычислений обратно модели.

Вместо этого мы просто разрешаем инструмент:

```python
tools=[{"type": "code_interpreter", "container": {"type": "auto"}}]
```

Контейнер будет создан автоматически на время запроса. Для демонстрации возьмём ту же таблицу продуктов в системном промпте, но теперь попросим не только посчитать БЖУ, а ещё и построить график — например, столбчатую диаграмму вклада ингредиентов в общую калорийность.


In [13]:
code_interpreter_instructions = f"""
Ты — ассистент-нутрициолог. Используй Code Interpreter для вычислений и визуализации.

В твоём распоряжении учебная таблица пищевой ценности на 100 граммов:

{nutrition_markdown}

Правила:
1. Для всех арифметических операций используй Python внутри Code Interpreter.
2. Используй только продукты из таблицы.
3. Итоговый ответ дай кратко и по делу.
4. Дополнительно построй аккуратную столбчатую диаграмму вклада каждого ингредиента в общую калорийность блюда.
5. Сохрани диаграмму как PNG-файл, чтобы её можно было скачать.
"""


user_says = (
    "Посчитай пищевую ценность обеда: 180 г риса варёного, "
    "160 г лосося и 120 г яблока. Затем построй график вклада ингредиентов в калории."
)

res = client.responses.create(
    model=model,
    instructions=code_interpreter_instructions,
    input=user_says,
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto",
            },
        }
    ],
)

printx(res.output_text)

Я рассчитаю пищевую ценность обеда из 180 г риса варёного, 160 г лосося и 120 г яблока, используя предоставленную таблицу пищевой ценности. Затем построю столбчатую диаграмму вклада каждого ингредиента в общую калорийность блюда.

Сначала я выполню необходимые вычисления с помощью Python, а затем создам визуализацию.

Обед содержит:

- **629.2 ккал**
- **37.2 г** белков
- **21.6 г** жиров
- **67.6 г** углеводов

Вклад ингредиентов в калорийность:
- Рис варёный (180 г): 234.0 ккал
- Лосось (160 г): 332.8 ккал
- Яблоко (120 г): 62.4 ккал

График вклада ингредиентов в калорийность построен и сохранён. Вы можете скачать его ниже.

[Скачать диаграмму](output/calorie_contribution.png)

В финальном тексте ответа модель уже покажет итоговую калорийность и БЖУ. Но, кроме текста, у нас есть и более подробная техническая информация:

* сам Python-код, который выполнил Code Interpreter;
* логи выполнения;
* аннотации со сгенерированными файлами.

Ниже мы пройдём по `response.output`, выведем код и попробуем скачать все артефакты в локальную директорию.


In [16]:
from pathlib import Path

def inspect_code_interpreter_response(response, download_dir='.'):
    download_dir = Path(download_dir)

    print(f"ID ответа: {response.id}")
    print("Типы элементов ответа:", [item.type for item in response.output])

    downloaded_files = []

    for item in response.output:
        if item.type == "code_interpreter_call":
            print("\n=== Code Interpreter ===")
            print("Container ID:", getattr(item, "container_id", None))
            print("\nКод, который выполнила модель:\n")
            print(item.code)

            for output_item in getattr(item, "outputs", []):
                output_type = getattr(output_item, "type", "unknown")
                logs = getattr(output_item, "logs", "")
                if logs:
                    print(f"\n[{output_type.upper()}] Вывод:")
                    print(logs)

        elif item.type == "message":
            for content in item.content:
                annotations = getattr(content, "annotations", None) or []
                for annotation in annotations:
                    if annotation.type == "container_file_citation":
                        file_id = annotation.file_id
                        filename = annotation.filename
                        local_path = download_dir / filename
                        try:
                            file_content = client.files.content(file_id)
                            with open(local_path, "wb") as file_handle:
                                file_handle.write(file_content.read())
                            downloaded_files.append(local_path)
                            print(f"\nСкачан файл: {local_path}")
                        except Exception as exc:
                            print(f"\nНе удалось скачать {filename}: {exc}")

    return downloaded_files


downloads = inspect_code_interpreter_response(res)

print(f"Скачаны файлы: {downloads}")

ID ответа: 27f7e105-f436-46e0-8bad-5cd766673ca8
Типы элементов ответа: ['message', 'code_interpreter_call', 'message']

=== Code Interpreter ===
Container ID: baa9bb66-64d9-4449-b7bf-7853721d11e0

Код, который выполнила модель:

import pandas as pd
import matplotlib.pyplot as plt

# Создание таблицы с данными
food_data = {
    'food': ['Куриная грудка', 'Яйцо куриное', 'Гречка варёная', 'Рис варёный', 'Овсянка сухая',
             'Лосось', 'Творог 5%', 'Банан', 'Яблоко', 'Картофель варёный', 'Огурец', 'Оливковое масло'],
    'kcal': [165, 143, 110, 130, 379, 208, 121, 89, 52, 87, 15, 884],
    'protein_g': [31.0, 12.6, 4.2, 2.7, 13.2, 20.0, 17.0, 1.1, 0.3, 1.9, 0.8, 0.0],
    'fat_g': [3.6, 9.5, 1.1, 0.3, 6.5, 13.0, 5.0, 0.3, 0.2, 0.1, 0.1, 100.0],
    'carbs_g': [0.0, 0.7, 21.3, 28.2, 67.7, 0.0, 3.0, 22.8, 14.0, 20.1, 3.0, 0.0]
}
df = pd.DataFrame(food_data)
df.set_index('food', inplace=True)

# Определение ингредиентов и их количества
ingredients = {
    'Рис варёный': 180,
    'Лос

## Пример 3. Файлы внутри Code Interpreter и явный контейнер

В предыдущем примере таблица продуктов по-прежнему жила в системном промпте. Это удобно для короткой демонстрации, но неудобно для более серьёзных сценариев:

* промпт становится длиннее;
* данные дублируются в каждом запросе;
* обновлять таблицу через текст неудобно.

Поэтому теперь перенесём каталог продуктов в настоящий CSV-файл и загрузим его в Code Interpreter через Files API. А затем создадим **явный контейнер** через `client.containers.create(...)`.

Преимущество явного контейнера в том, что его состояние можно использовать повторно в течение жизни контейнера. Это позволяет:

* хранить загруженные файлы без повторной отправки;
* создавать новые файлы внутри контейнера;
* продолжать работу в следующих запросах;
* поддерживать небольшой «рабочий контекст» между шагами.

В нашем случае контейнер будет хранить:

* исходный CSV-каталог продуктов;
* Excel-файл `food_log.xlsx` с дневником питания за день.


In [59]:
with open("food_catalog.csv", "rb") as file_handle:
    catalog_file = client.files.create(
        file=file_handle,
        purpose="assistants",
    )

container = client.containers.create(
    name="food-diary-demo",
    expires_after={"anchor": "last_active_at", "minutes": 20},
    file_ids=[catalog_file.id],
)

print("CSV-файл загружен с file_id:", catalog_file.id)
print("Создан container_id:", container.id)

CSV-файл загружен с file_id: fvt062cm35qkad2g8hre
Создан container_id: c565fc32-182d-4341-91de-41d87268b370


Теперь таблица продуктов уже не передаётся в промпте. Вместо этого модель должна найти файл `food_catalog.csv` внутри контейнера, прочитать его через pandas и использовать как источник данных. Кроме того, предположим, что мы хотим подсчитывать калории всех съеденных за день блюд. Для этого попросим модель создать в контейнере Excel-файл с дневником, и записывать туда все съеденные блюда.


In [60]:
foods = ', '.join(food_df['food'])

explicit_container_instructions = f"""
Ты — ассистент-нутрициолог, работающий внутри Code Interpreter. Пользователь будет сообщать тебе, какие блюда из каких ингредиентов он ест. Твоя задача - считать калории съеденного, и записывать статистику в дневник food_log.xlsx.

В контейнер уже загружен CSV-файл food_catalog.csv с колонками:
food, kcal, protein_g, fat_g, carbs_g

Пользователь сообщает тебе съеденное блюдо, твоя задача - посчитать калории и БЖУ и записать всё это в food_log.xslx. Если файла нет - создай его. Требования к файлу:
* столбцы: timestamp, meal_text, calories, protein_g, fat_g, carbs_g
* timestamp запиши в удобном текстовом формате

Доступные блюда:
{foods}

Правила:
1. Для поиска используй только названия доступных тебе блюд, при необходимости перефразируй.
2. Для поиска и вычислений читай данные из CSV-файла.
3. Не опирайся на память модели и не придумывай значения.
4. Для вычислений используй Python и pandas.
5. В ответе укажи итоговые калории и БЖУ.
"""

res = client.responses.create(
    model=model,
    instructions=explicit_container_instructions,
    input=(
        "Посчитай ужин: 200 г творога 5%, 120 г банана и 15 г оливкового масла."
    ),
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": container.id,
        }
    ],
)

printx(res.output_text)

Я рассчитаю калории и БЖУ для указанного ужина. Для этого мне нужно:

1. Проверить наличие файла food_catalog.csv с данными о продуктах
2. Загрузить данные о калорийности и БЖУ продуктов
3. Найти значения для творога 5%, банана и оливкового масла
4. Рассчитать пропорциональные значения для указанных весов
5. Суммировать итоговые калории и БЖУ
6. Записать данные в food_log.xlsx (создать файл, если его нет)

Начну с проверки доступных файлов.
Файл food_catalog.csv найден. Теперь я загружу данные из этого файла, рассчитаю калории и БЖУ для указанного ужина (200 г творога 5%, 120 г банана и 15 г оливкового масла), а затем запишу результаты в food_log.xlsx.

Я рассчитал калории и БЖУ для вашего ужина:

**Итоговое содержание питательных веществ:**
- Калории: 481.4 ккал
- Белки: 35.3 г
- Жиры: 25.4 г
- Углеводы: 33.4 г

**Состав ужина:**
- 200 г творога 5%: 242.0 ккал, Б: 34.0 г, Ж: 10.0 г, У: 6.0 г
- 120 г банана: 106.8 ккал, Б: 1.3 г, Ж: 0.4 г, У: 27.4 г
- 15 г оливкового масла: 132.6 ккал, Б: 0.0 г, Ж: 15.0 г, У: 0.0 г

Данные успешно записаны в файл food_log.xlsx, который был создан в директории output.

При желании можно так же посмотреть, какой именно код выполнил интерпретатор на первом шаге:


In [61]:
inspect_code_interpreter_response(res)


ID ответа: 6277b7bf-72f3-462f-adb3-09aa3175417f
Типы элементов ответа: ['message', 'message', 'code_interpreter_call', 'message']

=== Code Interpreter ===
Container ID: c565fc32-182d-4341-91de-41d87268b370

Код, который выполнила модель:

import pandas as pd
import numpy as np
from datetime import datetime
import os

def analyze_meal():
    try:
        # Проверяем и создаем output директорию
        if not os.path.exists('./output'):
            os.makedirs('./output')
            
        # Загружаем каталог продуктов
        food_catalog = pd.read_csv('./food_catalog.csv')
        
        # Определяем продукты и их веса
        meal_items = [
            {'food': 'Творог 5%', 'weight': 200},
            {'food': 'Банан', 'weight': 120},
            {'food': 'Оливковое масло', 'weight': 15}
        ]
        
        total_calories = 0
        total_protein = 0
        total_fat = 0
        total_carbs = 0
        
        # Рассчитываем калории и БЖУ для каждого продукта
        f

[WindowsPath('food_log.xlsx')]

Теперь используем тот же самый `container_id`, и говорим модели об очередном блюде. Code Interpreter будет поддерживать таблицу всех приёмов пищи за день внутри одного контейнера.


In [62]:
res1 = client.responses.create(
    model=model,
    previous_response_id=res.id,
    instructions=explicit_container_instructions,
    input="Я съел 150 грамм гречки",
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": container.id,
        }
    ],
)

printx(res1.output_text)


Я рассчитаю калории и БЖУ для 150 грамм варёной гречки и добавлю эту запись в ваш дневник питания food_log.xlsx.

Для начала загружу каталог продуктов и проверю существующий файл дневника.
Я рассчитал калории и БЖУ для 150 грамм варёной гречки:

**Итоговое содержание питательных веществ:**
- Калории: 165.0 ккал
- Белки: 6.3 г
- Жиры: 1.6 г
- Углеводы: 31.9 г

Данные успешно добавлены в ваш дневник питания food_log.xlsx. Файл был обновлён и сохранён в директории output.

В заключение мы можем попросить агента посчитать средние значение КБЖУ за день, используя **тот же самый контейнер**. Именно это и есть ключевая идея явного контейнера: мы не только выполняем код, но и сохраняем промежуточные артефакты между шагами одного сценария.

In [72]:
res2 = client.responses.create(
    model=model,
    previous_response_id=res1.id,
    instructions=explicit_container_instructions,
    input="""
    Используй Code Interpreter.
    Открой файл food_log.xlsx из текущего контейнера.
    Посчитай суммарные калории, белки, жиры и углеводы по каждому из дней, и количество
    приёмов пищи. Выведи табличку всех показателей по дням. 
    """,
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": container.id,
        }
    ],
)

printx(res2.output_text)

Я проанализирую файл food_log.xlsx и рассчитаю суммарные показатели по дням. Для этого мне нужно:

1. Проверить наличие файла food_log.xlsx
2. Загрузить данные из файла
3. Преобразовать timestamp в дату
4. Сгруппировать данные по дням и рассчитать суммы калорий, БЖУ и количество приёмов пищи
5. Вывести итоговую таблицу

Начну с проверки наличия файла.
Файл food_log.xlsx найден в директории output. Теперь я загружу данные из этого файла, обработаю их и рассчитаю суммарные показатели по дням.
Вот итоговая таблица суммарных показателей по дням:

| Дата | Калории (ккал) | Белки (г) | Жиры (г) | Углеводы (г) | Количество приёмов пищи |
|------|----------------|-----------|----------|--------------|------------------------|
| 2026-04-22 | 646.4 | 41.6 | 27.0 | 65.3 | 2 |

**Интерпретация данных:**
- В этот день у вас было 2 приёма пищи: ужин с творогом, бананом и оливковым маслом, а также приём гречки
- Суммарное потребление составило 646.4 ккал с распределением БЖУ: 41.6г белков, 27.0г жиров, 65.3г углеводов

Все данные успешно обработаны и подтверждены.

In [73]:
json.loads(res2.json())

C:\Users\dmitr\AppData\Local\Temp\ipykernel_59100\1766475823.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.loads(res2.json())


{'id': '431b72a7-f43c-4933-af1b-31627925c03f',
 'created_at': 1776895107.0,
 'error': None,
 'incomplete_details': None,
 'instructions': '\nТы — ассистент-нутрициолог, работающий внутри Code Interpreter. Пользователь будет сообщать тебе, какие блюда из каких ингредиентов он ест. Твоя задача - считать калории съеденного, и записывать статистику в дневник food_log.xlsx.\n\nВ контейнер уже загружен CSV-файл food_catalog.csv с колонками:\nfood, kcal, protein_g, fat_g, carbs_g\n\nПользователь сообщает тебе съеденное блюдо, твоя задача - посчитать калории и БЖУ и записать всё это в food_log.xslx. Если файла нет - создай его. Требования к файлу:\n* столбцы: timestamp, meal_text, calories, protein_g, fat_g, carbs_g\n* timestamp запиши в удобном текстовом формате\n\nДоступные блюда:\nКуриная грудка, Яйцо куриное, Гречка варёная, Рис варёный, Овсянка сухая, Лосось, Творог 5%, Банан, Яблоко, Картофель варёный, Огурец, Оливковое масло\n\nПравила:\n1. Для поиска используй только названия доступных

## Code Interpreter для анализа данных

Мы использовали интерпретатор кода для сравнительно простых задач, которые могли бы быть решены с помощью Tool Calling и специализированных инструментов. Но настоящая мощь интерпретатора кода проявляется тогда, когда модели дают возможность писать достаточно сложный код. Например, можно использовать Code Interpreter для анализа данных и построения предиктивных моделей.

Для примера возьмём [датасет ежедневных активностей пользователя](https://www.kaggle.com/datasets/aroojanwarkhan/fitness-data-trends). Предположим, мы хотим не только построить график активности пользователя, но и предсказать его активность в будущем. Табличка с активностями находится в файле [Activity.csv](Activity.csv).

Порядок наших действий такой же, как в предыдущем примере:

1. Загружаем файл `Activity.csv` с помощью File API
2. Создаём контейнер с находящимся внутри файлом
3. Используем Responses API с соответствующей инструкцией
4. Инспектируем ответ и скачиваем результаты

In [74]:
with open("Activity.csv", "rb") as file_handle:
    activity_file = client.files.create(
        file=file_handle,
        purpose="assistants",
    )

container = client.containers.create(
    name="data-analysis-demo",
    expires_after={"anchor": "last_active_at", "minutes": 20},
    file_ids=[activity_file.id],
)


Чтобы не ждать выполнения задания, а иметь возможность отслеживать процесс его выполнения - используем режим стриминга, указав параметр `streaming=True`. Он позволяет получать от модели периодические обновления, чтобы держать пользователя в курсе происходящего.

In [76]:
instructions = f"""
Ты — Data Scientist, анализирующий данные с помощью Code Interpreter. У тебя есть
файл Activity.csv с фитнес-активностью пользователя.
"""

stream = client.responses.create(
    model=model,
    instructions=instructions,
    input="""
        Тебе необходимо:

        1. Предварительно установить все модули Python, которые тебе могут понадобиться 
        2. Построить графики активности пользователя (`step_count`, `weight_kg` и `hours_of_sleep`) за прошлое время и предсказания его активностей на следующий месяц. Для каждого параметра - отдельный график. По возможности делай более интересные прогнозы, чем линейные.
        3. Построить график, показывающий наличие зависимости (корреляции) числа шагов `step_count` с настроением `mood`
    """,
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": container.id,
        }
    ],
    stream = True
)

resp_id = None
for event in stream:
    if event.type.endswith("delta"):
        print(event.delta, end='')
    elif event.type == "response.code_interpreter_call_code.done":
        print(f"\n\nВыполняемый код:\n{event.code}\n")
    elif event.type == "response.code_interpreter_call.in_progress":
        print("\n[Выполняем код...]\n")
    elif event.type == "response.code_interpreter_call.done":
        print("\n[Готово]\n")
    elif event.type == "response.in_progress":
        resp_id = event.response.id
        print(f"\n[Обрабатываем ответ {resp_id}]\n")


[Обрабатываем ответ 4f5efc12-d9a9-48bc-881e-2bca44254788]

Файл `Activity.csv` присутствует в рабочей директории. Также имеется файл `requirements.txt`, в котором, вероятно, указаны необходимые зависимости. Проверим его содержимое, чтобы убедиться, нужно ли дополнительно устанавливать какие-либо пакеты.

Но поскольку в инструкциях указано, что доступны `pandas`, `numpy`, `matplotlib`, а также можно устанавливать другие пакеты, я сразу начну с анализа данных и визуализации. Для более продвинутых прогнозов (не линейных) я использую временные ряды с трендом и сезонностью — например, экспоненциальное сглаживание (Holt-Winters) или SARIMA. Для простоты и надежности выберу **Exponential Smoothing**, так как он хорошо работает с небольшими наборами данных и может улавливать тренд и сезонность.

### План действий:
1. Установить необходимые библиотеки (если потребуется что-то сверху).
2. Прочитать данные из `Activity.csv`.
3. Провести предварительный анализ: проверить типы данных, пропуски, ди

Поскольку использовался режим стриминга, нам нужно сначала получить полный ответ, прежде чем скачать результаты исследований:

In [77]:
res = client.responses.retrieve(resp_id)
inspect_code_interpreter_response(res)

ID ответа: 4f5efc12-d9a9-48bc-881e-2bca44254788
Типы элементов ответа: ['message', 'code_interpreter_call', 'message', 'code_interpreter_call', 'code_interpreter_call', 'message', 'code_interpreter_call', 'message']

=== Code Interpreter ===
Container ID: 82f9fbe0-2cf3-43c0-8f03-12d0a35af219

Код, который выполнила модель:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import os

# Проверим, есть ли папка output, если нет — создадим
os.makedirs('./output', exist_ok=True)

# Шаг 1: Загрузка данных
try:
    df = pd.read_csv('Activity.csv')
    print("Файл успешно загружен.")
    print(f"Размер данных: {df.shape}")
    print("Первые 5 строк:")
    print(df.head())
    
    # Проверим типы данных и пропуски
    print("\nИнформация о данных:")
    print(df.info())
    
    print("\nПропущенные значения:")
    print(df.isnull().sum())
    
    # Преобразуем столбец даты
    df['date'] = pd.to_datetime(df['d

[WindowsPath('processed_data.csv'),
 WindowsPath('forecasts_next_month.csv'),
 WindowsPath('hours_of_sleep_forecast.png'),
 WindowsPath('step_count_forecast.png'),
 WindowsPath('weight_kg_forecast.png'),
 WindowsPath('correlation_step_mood.png')]

## Выводы

На одном и том же сценарии с расчётом калорий мы увидели сразу три уровня сложности:

* **Function Calling** подходит, когда вы хотите сами контролировать исполнение вычислений и полностью управлять бизнес-логикой приложения.
* **Code Interpreter с автоматическим контейнером** хорош для одноразовых вычислительных задач: посчитать значения, проверить данные, построить график и вернуть готовый артефакт.
* **Code Interpreter с явным контейнером и файлами** удобен для многошаговых сценариев, когда модель должна работать с CSV или XLSX, создавать новые файлы и продолжать вычисления в том же контейнере.

В практических приложениях эти механизмы не обязательно противопоставлять друг другу. Наоборот, очень часто они дополняют друг друга:

* внешние бизнес-действия, доступ к CRM или базе данных — через Function Calling;
* вычисления, обработка таблиц, графики и временные рабочие файлы — через Code Interpreter.

Важно:

* Function Calling является более детерминированной процедурой, поскольку мы заранее пишем код вызываемых функций. В реальной жизни для ведени дневника КБЖУ лучше было бы воспользоваться Function Calling, в этом примере мы используем Code Interpreter для демонстрации
* Code Interpreter - намного более гибкий, поскольку может генерировать произвольный код. Можно просить Code Interpreter анализировать данные, генерировать данные в сложных форматах (XLSX, PPTX и др.) и совершать любые действия, доступные из Python с учётом ограничений песочницы.  